In [4]:
import ast
import pandas as pd
import numpy as np

In [5]:
# data_folder = f'{data_folder}/umap25'
data_folder = f'.'

In [6]:
n_clusters = 128
tracks_distances = pd.read_csv(f'{data_folder}/tracks_percentage_distance_n_clusters{n_clusters}.csv')

In [7]:
tracks_distances.head()

,song_id,percentage_distance,labels
0,0,0.445913,126
1,1,-0.196602,38
2,3,0.351464,10
3,9,-0.405660,14
4,11,-0.006265,108


In [8]:
editorial_playlists = pd.read_csv(f'{data_folder}/editorial_playlists_integer_id.csv')
editorial_playlists.title = editorial_playlists.title.apply(lambda x: x.lower())

In [9]:
editorial_playlists.title.unique()

array(['dub essentials', 'emo essentials', 'pop essentials',
       'r&b essentials', 'folk essentials', 'funk essentials',
       'jazz essentials', 'punk essentials', 'rock essentials',
       'soul essentials', 'anime essentials', 'blues essentials',
       'dance essentials', 'disco essentials', 'divas essentials',
       'duets essentials', 'grime essentials', 'house essentials',
       'k-pop essentials', 'metal essentials', 'grunge essentials',
       'lounge essentials', 'reggae essentials', 'techno essentials',
       'trance essentials', 'ambient essentials', 'art pop essentials',
       'country essentials', 'dubstep essentials', 'acoustic essentials',
       'afrobeat essentials', 'boy band essentials',
       'brit pop essentials', 'festival essentials',
       'hardcore essentials', 'new wave essentials',
       'nu metal essentials', 'pop punk essentials',
       'rap rock essentials', 'sl house essentials',
       'ska punk essentials', 'slow jam essentials',
       'tr

In [10]:
selected = ['pop essentials', 'rock essentials', 'classical essentials', 'electronic essentials']

In [11]:
selected_editorial_playlists = editorial_playlists[editorial_playlists.title.isin(selected)][['title', 'song_id']]

In [12]:
selected_editorial_playlists.head(10)

,title,song_id
3,pop essentials,"[3418, 35738, 58468, 33270, 10581, 69789, 3100..."
12,rock essentials,"[56386, 63041, 49424, 75345, 52792, 23988, 285..."
70,classical essentials,"[48000, 12044, 52893, 71138, 43056, 48157, 667..."
71,classical essentials,"[65365, 59695, 12044, 52893, 71138, 43056, 481..."
87,electronic essentials,"[34634, 60012, 6982, 22757, 37145, 2402, 69847..."
88,electronic essentials,"[34634, 60012, 6982, 22757, 37145, 2402, 69847..."


In [13]:
selected_editorial_playlists.song_id = selected_editorial_playlists.song_id.apply(ast.literal_eval)

In [14]:
selected_editorial_playlists.song_id.apply(lambda x: len(x))

3     70
12    81
70    13
71    22
87    27
88    27
Name: song_id, dtype: int64

In [15]:
selected_editorial_playlists = selected_editorial_playlists.explode('song_id')

In [16]:
selected_editorial_playlists.groupby('title')['song_id'].nunique()

title
classical essentials     23
electronic essentials    27
pop essentials           70
rock essentials          81
Name: song_id, dtype: int64

In [17]:
# Some playlists are duplicated
selected_editorial_playlists = selected_editorial_playlists.drop_duplicates()

In [18]:
how_often_in_editorial = selected_editorial_playlists.groupby('song_id').count()

In [19]:
distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')

/tmp/ipykernel_264750/1329098343.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')


In [20]:
# Tracks that are in editorial playlists are closer to the cluster centroids
distances_for_playlists[['percentage_distance', 'title']].groupby('title').nunique()

,percentage_distance
title,
classical essentials,23
electronic essentials,18
none,26013
pop essentials,55
rock essentials,65


In [21]:
distances_for_playlists.head()

,song_id,percentage_distance,labels,title
0,0,0.445913,126,none
1,1,-0.196602,38,none
2,3,0.351464,10,none
3,9,-0.405660,14,none
4,11,-0.006265,108,electronic essentials


In [22]:
# Tracks that are in editorial playlists are closer to the cluster centroids
distances_for_playlists[['percentage_distance', 'title']].groupby('title').mean()

,percentage_distance
title,
classical essentials,-0.311813
electronic essentials,-0.071391
none,0.008685
pop essentials,-0.152567
rock essentials,-0.205352


In [23]:
# Check co-occurrence of cluster labels and editorial playlists

In [24]:
tracks_in_playlists = distances_for_playlists[['labels', 'title']]
tracks_in_playlists = tracks_in_playlists[tracks_in_playlists.title.isin(selected)]
tracks_in_playlists

,labels,title
4,108,electronic essentials
407,6,pop essentials
459,108,pop essentials
485,120,rock essentials
522,120,rock essentials
...,...,...
26638,108,pop essentials
27042,101,rock essentials
27052,91,pop essentials
27090,1,rock essentials


In [25]:
# out of the 128, only 22 are in the selected playlists, although there are 162 tracks
tracks_in_playlists.labels.nunique()

22

In [26]:
labels_for_playlist = tracks_in_playlists.groupby('title').agg(['unique'])
labels_for_playlist.columns = labels_for_playlist.columns.map('_'.join)

In [27]:
labels_for_playlist.head()

,labels_unique
title,
classical essentials,"[11, 45]"
electronic essentials,"[108, 16, 53, 102, 92, 9, 123, 91]"
pop essentials,"[6, 108, 91, 101, 83, 12, 48, 92, 10, 125, 53]"
rock essentials,"[120, 101, 92, 77, 1, 6, 26, 39, 16]"


In [28]:
# Jaccard similarity between editorial playlists, in terms of cluster labels
# Only restricting to number of clusters above the elbow, i.e., ns_clusters >=32
# Since for 128 all similarities are < 1/3, we can stop there

ns_clusters = [32, 64, 128]
for n_clusters in ns_clusters:
    tracks_distances = pd.read_csv(f'{data_folder}/tracks_percentage_distance_n_clusters{n_clusters}.csv')
    distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')

    tracks_in_playlists = distances_for_playlists[['labels', 'title']]
    tracks_in_playlists = tracks_in_playlists[tracks_in_playlists.title.isin(selected)]

    labels_for_playlist = tracks_in_playlists.groupby('title').agg(['unique'])
    labels_for_playlist.columns = labels_for_playlist.columns.map('_'.join)
    print(n_clusters)

    pair_of_selected = [(a, b) for idx, a in enumerate(selected) for b in selected[idx + 1:]]
    jaccards = []
    for title_1, title_2 in pair_of_selected:
        set_1 = labels_for_playlist.loc[title_1].labels_unique
        set_1 = set(set_1)
        set_2 = labels_for_playlist.loc[title_2].labels_unique
        set_2 = set(set_2)

        jaccard = len(set_1.intersection(set_2)) / len(set_1.union(set_2))
        jaccards += [jaccard]
        print(f'{title_1} ---- {title_2}: {jaccard}')
    jaccards = np.array(jaccards)
    print(f'{n_clusters} ---- {jaccards.mean()}')



32
pop essentials ---- rock essentials: 0.14285714285714285
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.42857142857142855
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.25
classical essentials ---- electronic essentials: 0.0
32 ---- 0.1369047619047619
64
pop essentials ---- rock essentials: 0.125
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.4
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.18181818181818182
classical essentials ---- electronic essentials: 0.0
64 ---- 0.1178030303030303
128
pop essentials ---- rock essentials: 0.17647058823529413
pop essentials ---- classical essentials: 0.0
pop essentials ---- electronic essentials: 0.26666666666666666
rock essentials ---- classical essentials: 0.0
rock essentials ---- electronic essentials: 0.13333333333333333
classical essentials ---- electronic essenti

/tmp/ipykernel_264750/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')
/tmp/ipykernel_264750/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  distances_for_playlists = tracks_distances.merge(selected_editorial_playlists, how='left', on='song_id').fillna('none')
/tmp/ipykernel_264750/1620312336.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a fu